# Week 1 — Data Loading, Cleaning & EDA
## Austin Animal Center · Outcomes + Intakes

**Goal:** Load both datasets from the Socrata API, fix all known data quality issues,
engineer baseline features, and explore the distributions that shape Week 2 modeling decisions.

**Binary target:** Adopted (1) vs Not Adopted (0)  
**Multi-class target:** Adopted / Transferred / Reclaimed / Euthanized / Died / Other

## 0 · Setup

Run the install cell once if any packages are missing, then **restart the kernel**.
Packages already installed (pandas, numpy) are safe to include — pip is idempotent.

In [ ]:
# Run this cell once if needed, then restart the kernel before continuing.
# !pip install pandas numpy matplotlib seaborn scikit-learn xgboost imbalanced-learn requests

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f'pandas  {pd.__version__}')
print(f'numpy   {np.__version__}')

pandas  3.0.0
numpy   2.4.1


### 2 · Full export CSVs (manual download)

Local copies of the full historical exports (2013-10-01 to 2025-05-05) downloaded
from the Austin data portal UI. Schema differs from the Socrata API
(Title Case + spaces vs snake_case), so they are kept as separate frames for
inspection only — not merged with the API-fetched data.

In [41]:
FULL_OUTCOMES_PATH = Path('../data/raw/Austin_Animal_Center_Outcomes_(10_01_2013_to_05_05_2025)_20260523.csv')
FULL_INTAKES_PATH  = Path('../data/raw/Austin_Animal_Center_Intakes_(10_01_2013_to_05_05_2025)_20260523.csv')

df_full_outcomes = pd.read_csv(FULL_OUTCOMES_PATH)
df_full_intakes  = pd.read_csv(FULL_INTAKES_PATH)

print('=== FULL OUTCOMES EXPORT ===')
df_full_outcomes.info()
print('\n--- head(3) ---')
display(df_full_outcomes.head(3))

print('\n=== FULL INTAKES EXPORT ===')
df_full_intakes.info()
print('\n--- head(3) ---')
df_full_intakes.head(3)

=== FULL OUTCOMES EXPORT ===
<class 'pandas.DataFrame'>
RangeIndex: 173775 entries, 0 to 173774
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   Animal ID         173775 non-null  str  
 1   Date of Birth     173775 non-null  str  
 2   Name              123991 non-null  str  
 3   DateTime          173775 non-null  str  
 4   MonthYear         173775 non-null  str  
 5   Outcome Type      173729 non-null  str  
 6   Outcome Subtype   79660 non-null   str  
 7   Animal Type       173775 non-null  str  
 8   Sex upon Outcome  173774 non-null  str  
 9   Age upon Outcome  173766 non-null  str  
 10  Breed             173775 non-null  str  
 11  Color             173775 non-null  str  
dtypes: str(12)
memory usage: 15.9 MB

--- head(3) ---


,Animal ID,Date of Birth,Name,DateTime,MonthYear,Outcome Type,Outcome Subtype,Animal Type,Sex upon Outcome,Age upon Outcome,Breed,Color
0,A668305,2012-12-01,NaN,2013-12-02T00:00:00-05:00,12-2013,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Brown/Yellow
1,A673335,2012-02-22,NaN,2014-02-22T00:00:00-05:00,02-2014,Euthanasia,Suffering,Other,Unknown,2 years,Raccoon,Black/Gray
2,A675999,2013-04-03,NaN,2014-04-07T00:00:00-05:00,04-2014,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Green



=== FULL INTAKES EXPORT ===
<class 'pandas.DataFrame'>
RangeIndex: 173812 entries, 0 to 173811
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   Animal ID         173812 non-null  str  
 1   Name              123821 non-null  str  
 2   DateTime          173812 non-null  str  
 3   MonthYear         173812 non-null  str  
 4   Found Location    173812 non-null  str  
 5   Intake Type       173812 non-null  str  
 6   Intake Condition  173812 non-null  str  
 7   Animal Type       173812 non-null  str  
 8   Sex upon Intake   173811 non-null  str  
 9   Age upon Intake   173812 non-null  str  
 10  Breed             173812 non-null  str  
 11  Color             173812 non-null  str  
dtypes: str(12)
memory usage: 15.9 MB

--- head(3) ---


,Animal ID,Name,DateTime,MonthYear,Found Location,Intake Type,Intake Condition,Animal Type,Sex upon Intake,Age upon Intake,Breed,Color
0,A521520,Nina,10/01/2013 07:51:00 AM,October 2013,Norht Ec in Austin (TX),Stray,Normal,Dog,Spayed Female,7 years,Border Terrier/Border Collie,White/Tan
1,A664235,NaN,10/01/2013 08:33:00 AM,October 2013,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White
2,A664236,NaN,10/01/2013 08:33:00 AM,October 2013,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White


### 3 · Clean + merge full-export CSVs into one frame

Align the 173k historical CSV rows to a snake_case schema, then backward-asof
merge intakes ← outcomes so each outcome carries its triggering intake's
context. Output: `df_full_merged` — one row per outcome event.

In [47]:
# --- outcomes side: rename to target schema ---
o = df_full_outcomes.rename(columns={
    'Animal ID':       'animal_id',
    'DateTime':        'outcome_date',
    'Date of Birth':   'date_of_birth',
    'Outcome Type':    'outcome_type',
    'Outcome Subtype': 'outcome_subtype',
    'Animal Type':     'animal_type',
    'Color':           'color_raw',       # to be split in cell 4
    'Sex upon Outcome':'sex_upon_outcome',# to be split in cell 4
})

o = o.drop(columns=['MonthYear', 'Age upon Outcome', 'Breed', 'Name'])

# --- intakes side: rename to target schema ---
i = df_full_intakes.rename(columns={
    'Animal ID':        'animal_id',
    'DateTime':         'intake_date',
    'Found Location':   'found_location',
    'Intake Type':      'intake_reason',
    'Intake Condition': 'intake_health_condition',
    'Breed':            'primary_breed',
    'Sex upon Intake':  'sex_upon_intake', # to be processed in cell 4
})

i = i.drop(columns=['Animal Type', 'Color', 'MonthYear', 'Age upon Intake', 'Name'])

print(f'After rename — outcomes: {o.shape}, intakes: {i.shape}')
print('Outcomes cols:', list(o.columns))
print('Intakes  cols:', list(i.columns))

After rename — outcomes: (173775, 8), intakes: (173812, 7)
Outcomes cols: ['animal_id', 'date_of_birth', 'outcome_date', 'outcome_type', 'outcome_subtype', 'animal_type', 'sex_upon_outcome', 'color_raw']
Intakes  cols: ['animal_id', 'intake_date', 'found_location', 'intake_reason', 'intake_health_condition', 'sex_upon_intake', 'primary_breed']


In [59]:
# Helper: parse mixed ISO datetimes into date-only (YYYY-MM-DD)
# Reasons:
# - Source strings contain both timezone-aware (e.g. 2013-12-02T00:00:00-05:00)
#   and naive ISO strings (e.g. 2013-10-01T09:31:00).
# - If pandas parses a Series as tz-aware, the naive strings can become NaT.
# - To avoid that, strip the timezone offset suffix, parse, then normalize to date.

def _parse_date_only(series: pd.Series) -> pd.Series:
    """Return a date-only Series (datetime64[ns]) from mixed ISO datetime strings."""
    series = series.astype(str).str.strip()
    # Remove timezone offsets like -05:00 or +00:00 so both offset and naive strings parse
    series = series.str.replace(r'([+-]\d{2}):(\d{2})$', '', regex=True)
    # Parse; errors='coerce' turns unparseable values into NaT, .dt.normalize() keeps date only
    return pd.to_datetime(series, errors='coerce').dt.normalize()

# Apply parsing to columns (works whether the column currently holds raw strings
# or Timestamp objects represented as strings)
o['outcome_date'] = _parse_date_only(o['outcome_date'])
o['date_of_birth'] = _parse_date_only(o['date_of_birth'])
i['intake_date'] = _parse_date_only(i['intake_date'])

# DOB after outcome is impossible — null these (compare dates only)
mask = o['date_of_birth'] > o['outcome_date']
o.loc[mask, 'date_of_birth'] = pd.NaT

# Audits
print(f'Nulled impossible DOBs: {o["date_of_birth"].isna().sum():,}')
print(o['outcome_date'].info(),i['intake_date'].info())

Nulled impossible DOBs: 33
<class 'pandas.Series'>
RangeIndex: 173775 entries, 0 to 173774
Series name: outcome_date
Non-Null Count   Dtype         
--------------   -----         
173775 non-null  datetime64[us]
dtypes: datetime64[us](1)
memory usage: 1.3 MB
<class 'pandas.Series'>
RangeIndex: 173812 entries, 0 to 173811
Series name: intake_date
Non-Null Count   Dtype         
--------------   -----         
173812 non-null  datetime64[us]
dtypes: datetime64[us](1)
memory usage: 1.3 MB
None None


### Parse sex, spay/neuter & color fields

Split "Sex upon Outcome" / "Sex upon Intake" into sex + spayed_neutered flags, split "Color" into primary/secondary colors, derive boolean ispreviouslyspayedneutered from intake sex, and drop the raw source columns.

In [ ]:
SEX_UPON_MAP = {
    'Neutered Male': ('Male',    'Yes'),
    'Spayed Female': ('Female',  'Yes'),
    'Intact Male':   ('Male',    'No'),
    'Intact Female': ('Female',  'No'),
    'Unknown':       ('Unknown', 'Unknown'),
}


def _split_sex(series: pd.Series) -> tuple[pd.Series, pd.Series]:
    mapped = series.map(SEX_UPON_MAP)
    sex  = mapped.map(lambda t: t[0] if isinstance(t, tuple) else 'Unknown')
    spay = mapped.map(lambda t: t[1] if isinstance(t, tuple) else 'Unknown')
    return sex, spay


def _split_slash(series: pd.Series) -> tuple[pd.Series, pd.Series]:
    parts = series.fillna('').str.split('/', n=1, expand=True)
    primary   = parts[0].replace('', np.nan)
    secondary = parts[1].replace('', np.nan) if parts.shape[1] > 1 else np.nan
    return primary, secondary


# --- outcomes: split Sex upon Outcome -> sex + spayed_neutered ---
o['sex'], o['spayed_neutered'] = _split_sex(o['sex_upon_outcome'])
o['primary_color'], o['secondary_color'] = _split_slash(o['color_raw'])
o = o.drop(columns=['sex_upon_outcome', 'color_raw'])

# --- intakes: derive ispreviouslyspayedneutered (bool) from Sex upon Intake ---
_, spay_intake = _split_sex(i['sex_upon_intake'])
i['ispreviouslyspayedneutered'] = spay_intake.map({'Yes': True, 'No': False}).astype('boolean')
i = i.drop(columns=['sex_upon_intake'])

outcomes sex distribution:
sex
Male       83196
Female     77073
Unknown    13506

outcomes spayed_neutered:
spayed_neutered
Yes        116202
No          44067
Unknown     13506

intakes ispreviouslyspayedneutered:
ispreviouslyspayedneutered
False    115710
True      44560
<NA>      13542


In [57]:
intake_cols = [
    'animal_id', 'intake_date',
    'intake_reason', 'intake_health_condition', 'found_location',
    'ispreviouslyspayedneutered', 'primary_breed',
]

intakes_for_merge = (
    i[intake_cols]
    .sort_values('intake_date')
)
outcomes_sorted = o.sort_values('outcome_date')

df_full_merged = pd.merge_asof(
    outcomes_sorted,
    intakes_for_merge,
    by='animal_id',
    left_on='outcome_date',
    right_on='intake_date',
    direction='backward',
)

# Derived columns
df_full_merged['length_of_stay_days'] = (
    df_full_merged['outcome_date'] - df_full_merged['intake_date']
).dt.days
df_full_merged['age_at_intake_days'] = (
    df_full_merged['intake_date'] - df_full_merged['date_of_birth']
).dt.days
df_full_merged['is_adopted'] = (df_full_merged['outcome_type'] == 'Adoption').astype(int)

# Audits
print(f'Row parity (merged vs outcomes): {len(df_full_merged):,} vs {len(o):,}')
print(f'Missing intake_date:             {df_full_merged["intake_date"].isna().sum():,}')
print(f'Negative LOS:                    {(df_full_merged["length_of_stay_days"] < 0).sum():,}')
print(f'Median LOS:                      {df_full_merged["length_of_stay_days"].median():.0f} days')

Row parity (merged vs outcomes): 173,775 vs 173,775
Missing intake_date:             925
Negative LOS:                    0
Median LOS:                      6 days


In [ ]:
processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

out_path = processed_dir / 'df_full_merged.csv'
df_full_merged.to_csv(out_path, index=False)

print(f'Saved {len(df_full_merged):,} rows to {out_path}')